# Variational Classifier

https://pennylane.ai/qml/demos/tutorial_variational_classifier

Variational quantum classifiers are quantum circuits that can be trained from labelled data to classify new samples (i.e. a supervised machine learning algorithm)

The first example is based on [this paper](https://arxiv.org/abs/1802.06002) which shows that a variational circuit can be optimized to emulate the parity function:

$f : x \in \{0,1\}^{\otimes n} \rightarrow y = 
\begin{cases} 
1 & \text{if uneven number of ones in x} \\
0 & \text{else}
\end{cases}$

Using this we will encode binary inputs into the initial state of the variational circuit, i.e. encode it into a computational basis state

The second example encodes real vectors as amplitude vectors in quantum states, and trains a variational circuit to recognize the first two classes of the iris dataset

## 1. Fitting the parity function

In [1]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import NesterovMomentumOptimizer

In [2]:
#create a quantum device to run the circuits
dev = qml.device('default.qubit')

Variational classifiers usually define a "layer" or "block", i.e. an elementary circuit architecture that gets repeated to build the full variational circuit

In [6]:
#this circuit layer uses four qubits (wires) with arbitrary rotation
# on every qubit, and CNOT gates to entangle each qubits with its neighbors
# we call the layer parameters "weights"

def layer(layer_weights):
    for wire in range(4):
        qml.Rot(*layer_weights[wire], wires=wire)
    
    for wires in ([0,1],[1,2],[2,3],[3,0]):
        qml.CNOT(wires=wires)

We need a way to encode data inputs $x$ into the circuit in order to make the measured outpout depend on the inputs.

Here, inputs are bitstrings which we encode in the qubit state $\ket{\psi}$

$x=0101 \rightarrow \ket{\psi} = \ket{0101}$

PennyLane has a `BasisState` function made to do this. It expects `x` to be a list of 0s and 1s

In [7]:
def state_preparation(x):
    qml.BasisState(x, wires=[0,1,2,3])

In [8]:
#now define the variational quantum circuit as a 
# state preparation routine, followed by a repetition of the layer structure
@qml.qnode(dev)
def circuit(weights, x):

    state_preparation(x)

    for layer_weights in weights:
        layer(layer_weights)

    return qml.expval(qml.PauliZ(0))

In [9]:
#we can add a "classical" bias parameter, but then
# the variational quantum classifier also needs post-processing
#the full model is a sum of the quantum circuit output plus trainable bias
def variational_classifier(weights, bias, x):
    return circuit(weights, x) + bias

#### Cost

Now we add in some cost functions, usually the sum of a loss function and a regularizer